<a href="https://colab.research.google.com/github/nanaaries313/Portfolio/blob/main/NB_Model_Prediction_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Set up**
Install Packgages

In [6]:
#Store e1071 into pkgs
pkgs <- c("e1071")

#identify which packages from pkgs that are not currently installed
#Compare the packgage name with the pkgs rownames/ list
to_install <- pkgs[!pkgs %in% rownames(installed.packages())]

#if the length of to_install list >0, install the packages
if (length(to_install) > 0) install.packages(to_install)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘proxy’




To use the package





In [7]:
library (e1071)

#**Read data**

In [9]:
df <- read.csv("Weather.csv", stringsAsFactors = TRUE)

In [10]:
df$Humidity

[1] High    High    High    High    Normal  Normal  Normal  High    Normal 
[10] Normal  Normal  High    Normal  High   
Levels: High  Normal

In [12]:
str(df)

'data.frame':	14 obs. of  5 variables:
 $ Outlook : Factor w/ 3 levels "Overcast ","Rainy ",..: 3 3 1 2 2 2 1 3 3 2 ...
 $ Temp.   : Factor w/ 3 levels "Cool ","Hot ",..: 2 2 2 3 1 1 1 3 1 3 ...
 $ Humidity: Factor w/ 2 levels "High ","Normal ": 1 1 1 1 2 2 2 1 2 2 ...
 $ Windy   : Factor w/ 2 levels "FALSE ","TRUE ": 1 2 1 1 1 2 2 1 1 1 ...
 $ Play    : Factor w/ 2 levels "No ","Yes ": 1 1 2 2 2 1 2 1 2 2 ...


In [14]:
View(df)

Outlook,Temp.,Humidity,Windy,Play
<fct>,<fct>,<fct>,<fct>,<fct>
Sunny,Hot,High,FALSE,No
Sunny,Hot,High,TRUE,No
Overcast,Hot,High,FALSE,Yes
Rainy,Mild,High,FALSE,Yes
Rainy,Cool,Normal,FALSE,Yes
Rainy,Cool,Normal,TRUE,No
Overcast,Cool,Normal,TRUE,Yes
Sunny,Mild,High,FALSE,No
Sunny,Cool,Normal,FALSE,Yes


Print out the numbers of rows and columns as well as column names

In [15]:
#cat (concatenate and print) is used when want more control over the conntent
cat("Data dimensions:", nrow(df), "rows x", ncol (df), "cols\n\n")
cat ("Column names: \n")

#will have [1]
print (names(df))
cat ("\n")

Data dimensions: 14 rows x 5 cols

Column names: 
[1] "Outlook"  "Temp."    "Humidity" "Windy"    "Play"    



#**Contigency Tables**

In [17]:
cat("==============================================\n")
cat("Contigency tables: first four variables vs play\n")
cat("==============================================\n\n")

for (i in 1:4) {
  varname <- names (df) [i]
  cat (sprintf("----%s vs play ----\n", varname))
  print(table (df[[i]], df$Play, useNA = "ifany"))
  cat ("\n")
}

Contigency tables: first four variables vs play

----Outlook vs play ----
           
            No  Yes 
  Overcast    0    4
  Rainy       2    3
  Sunny       3    2

----Temp. vs play ----
       
        No  Yes 
  Cool    1    3
  Hot     2    2
  Mild    2    4

----Humidity vs play ----
         
          No  Yes 
  High      4    3
  Normal    1    6

----Windy vs play ----
        
         No  Yes 
  FALSE    2    6
  TRUE     3    3



Another way

In [18]:
table (df$Outlook, df$Play)

           
            No  Yes 
  Overcast    0    4
  Rainy       2    3
  Sunny       3    2

#**Train Naive Bayes Model**

In [20]:
#Play is the variable that I want to predict
#. means use all other variables as predictors
#Naive Bayes model should be trained using df dataframe
nb_model <- naiveBayes(Play~.,data=df)

#Access and display apriori probabilities
nb_model$apriori

#Access and display the conditional probability tables for the Outlook variable
nb_model$tables$Outlook

#Attempt to convert probablities to raw counts
nb_model$tables$Outlook*14

#Multiply first row of the Oulook conditional probability table by 5 for raw count
nb_model$tables$Outlook[1,]*5

#Multiply second row by 9 for raw count
nb_model$tables$Outlook[2,]*9

Y
 No  Yes  
   5    9 

      Outlook
Y      Overcast     Rainy     Sunny 
  No   0.0000000 0.4000000 0.6000000
  Yes  0.4444444 0.3333333 0.2222222

      Outlook
Y      Overcast    Rainy    Sunny 
  No    0.000000 5.600000 8.400000
  Yes   6.222222 4.666667 3.111111

Overcast     Rainy     Sunny  
        0         2         3

Overcast     Rainy     Sunny  
        4         3         2

Use the model to predict the training data

In [21]:
weather.predictions = predict (nb_model, df)

Create a contigency table of the predictions and the actual class values (confusion matrix)

In [22]:
table (df$Play,weather.predictions)

      weather.predictions
       No  Yes 
  No     4    1
  Yes    0    9

The error rate is 1/14

In [24]:
df.2 =cbind(df, weather.predictions)
View (df.2)

Outlook,Temp.,Humidity,Windy,Play,weather.predictions
<fct>,<fct>,<fct>,<fct>,<fct>,<fct>
Sunny,Hot,High,FALSE,No,No
Sunny,Hot,High,TRUE,No,No
Overcast,Hot,High,FALSE,Yes,Yes
Rainy,Mild,High,FALSE,Yes,Yes
Rainy,Cool,Normal,FALSE,Yes,Yes
Rainy,Cool,Normal,TRUE,No,Yes
Overcast,Cool,Normal,TRUE,Yes,Yes
Sunny,Mild,High,FALSE,No,No
Sunny,Cool,Normal,FALSE,Yes,Yes


#**Predict Class for New Data**

The basic data structure in R is called a data frame.

We can also create a data frame with the data we want to predict

In [26]:
weather.today = data.frame (
  Outlook = "Sunny",
  Temp. = "Mild",
  Humidity = "High",
  Windy = "TRUE"
)

Explore the data frame

Use the predict function to predict the class

In [27]:
predict(nb_model,weather.today)

[1] Yes 
Levels: No  Yes

Changing the type to "raw" outputs the probabilities instead of class predictions

In [29]:
predict (nb_model, weather.today, type = "raw")

No,Yes
0.3571429,0.6428571
